# **This notebook will consist of combining data from several sources to curate a low VP test set**

In [ ]:
import pandas as pd

df = pd.read_csv('1960s_NBoS_with_SMILES.csv')

import numpy as np

# -----------------------
# Detect pressure columns
# -----------------------

pressure_cols = []
for col in df.columns:
    try:
        float(col)
        pressure_cols.append(col)
    except:
        pass

pressure_cols = sorted(pressure_cols, key=float)

bad_compounds = []

# -----------------------
# Monotonicity check
# -----------------------

for idx, row in df.iterrows():

    temps = row[pressure_cols].astype(float)

    temps_clean = temps.dropna().values

    if len(temps_clean) < 2:
        continue

    if not np.all(np.diff(temps_clean) > 0):
        bad_compounds.append(
            (row["Formula"], row["Name"], temps_clean)
        )

# -----------------------
# Output
# -----------------------

print(f"\nCompounds with non-monotonic vapor pressure trends: {len(bad_compounds)}\n")

for formula, name, temps in bad_compounds:
    print(f"{formula} — {name}")
    print("Temperatures:", temps)
    print("-" * 60)



## **Processing the Datasets**


The data from the 1960s dataset yield pressure in **mmHg** (approx. **torr**), but the data from the Yaw's handbook that were obtained from the explosives paper yield pressure in **atmospheres**, so we need to account for that.

$$\log_{10}P = A - \frac{B}{T + C}$$
$$P\quad[=] \quad\text[{atm}]\quad[\approx]\quad\text[{760*torr}]$$
$$T \quad[=] \quad[^\circ\text{C}]$$

In [ ]:
import pandas as pd

df_lowVP_solids = pd.read_csv('./Useable Data/Hughes_Lias_Low_VP_Solids')
df_lowVP_liquids = pd.read_csv('./Useable Data/Hughes_Lias_Low_VP_Liquids')
# df_lowVP_Yaws = pd.read_csv('./DATASET_OF_ENTRIES_WITH_VALID_SMILES_MISSING_FROM_EXPLOSIVES_PAPER')

In [ ]:
print("Number of solid compounds =", len(df_lowVP_solids))
print("Number of liquid compounds =", len(df_lowVP_liquids))
display(df_lowVP_solids.head())
display(df_lowVP_liquids.head())

# display(df_lowVP_Yaws.head())

## **Structuring the Data for my pre-established Chemprop pipeline**

### Restructuring the Dataframe to Align with already prepared code

In [ ]:
keep_cols = ["Name", "Formula", "smiles", "A", "B", "C", "Tmin_C", "Tmax_C"]
df_lowVP_solids = df_lowVP_solids[keep_cols].copy()
df_lowVP_liquids = df_lowVP_liquids[keep_cols].copy()

df_lowVP_solids = df_lowVP_solids.rename(columns = {'smiles':'SMILES', 'Tmin_C':'TMIN', 'Tmax_C':'TMAX'})
df_lowVP_liquids = df_lowVP_liquids.rename(columns = {'smiles':'SMILES','Tmin_C':'TMIN', 'Tmax_C':'TMAX'})

display(df_lowVP_liquids.head())
display(df_lowVP_solids.head())

### Function for cleaning a dataset consisting of smiles strings

Removing ions, inorganics, organometallics, compounds with rare elements occuring is a small fraction of the data, etc.

In [ ]:
import pandas as pd
from rdkit import Chem
from collections import Counter
import os

def clean_smiles_dataset(df, output_path, threshold=0.01):
    """
    Cleans a dataset containing SMILES strings:
      - Canonicalizes SMILES
      - Removes invalid molecules
      - Removes inorganic molecules (no carbon)
      - Removes organometallic molecules
      - Removes ions
      - Removes molecules with > 30 heavy atoms
      - Drops duplicates
      - Removes molecules with elements appearing in < threshold fraction of dataset
      - Saves cleaned DataFrame as CSV with '_clean' suffix

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe containing a SMILES column ("SMILES" or "smiles").
    output_path : str
        Path to save the cleaned CSV (suffix '_clean.csv' will be added automatically).
    threshold : float
        Minimum fraction of molecules required for an element to be kept (default=0.01 = 1%).

    Returns
    -------
    cleaned_df : pd.DataFrame
        The cleaned DataFrame.
    atom_summary : pd.DataFrame
        Summary of atom counts across dataset.
    """
    # Detect column
    smiles_col = None
    for candidate in ["SMILES", "smiles"]:
        if candidate in df.columns:
            smiles_col = candidate
            break
    if smiles_col is None:
        raise ValueError("No 'SMILES' or 'smiles' column found in DataFrame.")
    
    # --- Helpers ---
    def canonicalize_smiles(smiles):
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return None
            return Chem.MolToSmiles(mol, canonical=True)
        except:
            return None

    def is_organic(mol):
        return any(atom.GetSymbol() == "C" for atom in mol.GetAtoms())

    def is_organometallic(mol):
        metal_atomic_numbers = set([
            3,4,11,12,13,19,20,21,22,23,24,25,26,27,28,29,30,
            31,33,37,38,39,40,41,42,43,44,45,46,47,48,49,50,
            55,56,57,72,73,74,75,76,77,78,79,80,81,82,83,
            87,88,89
        ])
        return any(atom.GetAtomicNum() in metal_atomic_numbers for atom in mol.GetAtoms())

    def is_ion(mol):
        return Chem.GetFormalCharge(mol) != 0

    def too_many_heavy_atoms(mol, max_heavy_atoms=30):
        return mol.GetNumHeavyAtoms() > max_heavy_atoms

    def analyze_elements(df, smiles_column):
        atom_type_counts = Counter()
        molecule_atom_presence = Counter()
        for smi in df[smiles_column]:
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                continue
            mol = Chem.AddHs(mol)
            atom_symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
            atom_type_counts.update(atom_symbols)
            molecule_atom_presence.update(set(atom_symbols))
        return pd.DataFrame({
            "Total_Atom_Count": pd.Series(atom_type_counts),
            "Molecule_Count": pd.Series(molecule_atom_presence)
        }).fillna(0).astype(int)

    def remove_rare_elements(df, smiles_col, threshold):
        atom_summary = analyze_elements(df, smiles_column=smiles_col)
        total_molecules = len(df)
        rare_elements = atom_summary[
            atom_summary["Molecule_Count"] / total_molecules < threshold
        ].index.tolist()
        print(f"Rare elements (<{threshold*100:.1f}% of molecules): {rare_elements}")
        mask = []
        for smi in df[smiles_col]:
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                mask.append(False)
                continue
            atoms = {atom.GetSymbol() for atom in mol.GetAtoms()}
            mask.append(not any(el in atoms for el in rare_elements))
        filtered_df = df[mask].reset_index(drop=True)
        print(f"Removed {len(df) - len(filtered_df)} molecule(s) due to rare elements.")
        return filtered_df, atom_summary

    # --- Filtering loop ---
    stats = {
        "invalid": [],
        "inorganic": [],
        "organometallic": [],
        "too_large": [],
        "ion": [],
        "kept": []
    }

    for _, row in df.iterrows():
        smi = row[smiles_col]
        cano = canonicalize_smiles(smi)
        if cano is None:
            stats["invalid"].append(row)
            continue

        mol = Chem.MolFromSmiles(cano)
        if mol is None:
            stats["invalid"].append(row)
            continue

        if not is_organic(mol):
            row[smiles_col] = cano
            stats["inorganic"].append(row)
            continue

        if is_organometallic(mol):
            row[smiles_col] = cano
            stats["organometallic"].append(row)
            continue

        if too_many_heavy_atoms(mol, max_heavy_atoms=30):
            row[smiles_col] = cano
            stats["too_large"].append(row)
            continue

        if is_ion(mol):
            row[smiles_col] = cano
            stats["ion"].append(row)
            continue

        row[smiles_col] = cano
        stats["kept"].append(row)

    # Build cleaned DataFrame
    cleaned_df = pd.DataFrame(stats["kept"]).drop_duplicates(subset=[smiles_col], keep="first")

    # Rare element filtering
    cleaned_df, atom_summary = remove_rare_elements(cleaned_df, smiles_col, threshold)

    import inspect

    # --- Saving block ---
    frame = inspect.currentframe().f_back
    df_name = None
    for var_name, var_val in frame.f_locals.items():
        if var_val is df:
            df_name = var_name
            break

    if df_name is None:
        df_name = "dataset"

    if not os.path.isdir(output_path):
        os.makedirs(output_path, exist_ok=True)

    output_file = os.path.join(output_path, f"{df_name}_clean.csv")
    cleaned_df.to_csv(output_file, index=False)
    print(f"✅ Cleaned dataset saved to {output_file}")

    # Report
    print("Filtering summary:")
    for k in ["invalid", "inorganic", "organometallic", "too_large", "ion", "kept"]:
        print(f"{k}: {len(stats[k])}")

    print(f"Original size: {len(df)}")
    print(f"After cleaning & rare element filter: {len(cleaned_df)}")

    return cleaned_df, atom_summary


### Perform the Cleansing

Here I opted to not remove compounds identified as having 'rare' elements simply because the rare elements were things like chlorine and fluorine. Still removed organometallics, and the dataset was already pure of inorganics.

In [ ]:
output_dir = "../Datasets/Curated Dataset(s)"
print('----------------------------------------------------------------')
print('               Results from Cleaning the Liquid Dataset')
print('----------------------------------------------------------------')
df_lowVP_liquids_cleaned, atom_summary1 = clean_smiles_dataset(df_lowVP_liquids, threshold=0, output_path=output_dir)
# print('----------------------------------------------------------------\n----------------------------------------------------------------')
print('----------------------------------------------------------------')
print('               Results from Cleaning the Solid Dataset')
print('----------------------------------------------------------------')
df_lowVP_solids_cleaned, atom_summary2 = clean_smiles_dataset(df_lowVP_solids, threshold=0, output_path=output_dir)

### Looking at the data distrbutions to see how they stack up with the training data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem

# Example: assume you already have a dataframe with a SMILES column
# df = pd.read_csv("your_data.csv")

def count_heavy_atoms(smiles: str) -> int:
    """Return the number of heavy (non-hydrogen) atoms in a SMILES string."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  # handle invalid SMILES gracefully
    return mol.GetNumHeavyAtoms()

# Add new column
df_lowVP_liquids_cleaned["heavy_atoms"] = df_lowVP_liquids_cleaned["SMILES"].apply(count_heavy_atoms)
df_lowVP_solids_cleaned["heavy_atoms"] = df_lowVP_solids_cleaned["SMILES"].apply(count_heavy_atoms)

# Plot histogram
plt.hist(df_lowVP_liquids_cleaned["heavy_atoms"].dropna(), bins=30, edgecolor="black")
plt.xlabel("Number of Heavy Atoms")
plt.ylabel("Count")
plt.title(r"Distribution of Heavy Atoms in Molecules" "\n" r"$\bf{-\ LOW\ VP\ Liquids\ Test\ Data\ -}$")
plt.show()

# Plot histogram
plt.hist(df_lowVP_solids_cleaned["heavy_atoms"].dropna(), bins=40, edgecolor="black")
plt.xlabel("Number of Heavy Atoms")
plt.ylabel("Count")
plt.title(r"Distribution of Heavy Atoms in Molecules" "\n" r"$\bf{-\ LOW\ VP\ Solids\ Test\ Data\ -}$")
plt.show()

## **Generating VPs at Different Temps and Preparing the File for Evaluation by the Chemprop Model**

### Function for Generating Vapor Pressures

In [ ]:
import numpy as np
import pandas as pd

def generate_vapor_pressures_augmented(
    df,
    num_points=30,
    seed=None,
    verbose=True
):
    if seed is not None:
        np.random.seed(seed)

    expanded_rows = []

    for idx, row in df.iterrows():

        A, B, C = row["A"], row["B"], row["C"]
        T_min = row["TMIN"]
        T_max = row["TMAX"]

        # -------------------
        # Basic validity checks
        # -------------------
        if not np.isfinite([A, B, C, T_min, T_max]).all():
            if verbose:
                print(f"Row {idx} skipped: NaN or inf values.")
            continue

        if T_max <= T_min:
            if verbose:
                print(f"Row {idx} skipped: invalid temp bounds ({T_min}, {T_max}).")
            continue

        # -------------------
        # Sample temperature range
        # -------------------
        T_range = np.random.uniform(
            low=T_min,
            high=T_max,
            size=num_points
        )

        # -------------------
        # Compute vapor pressures
        # -------------------
        for T_C in T_range:

            log10_P = A - B / (T_C + C)

            if not np.isfinite(log10_P):
                continue

            row_dict = row.to_dict()
            row_dict["T_C"] = T_C
            row_dict["log10_P"] = log10_P

            expanded_rows.append(row_dict)

    expanded_df = pd.DataFrame(expanded_rows)

    if verbose:
        print(f"Generated {len(expanded_df)} samples from {len(df)} molecules")

    return expanded_df


### Generating VPs and Formatting the Files

In [ ]:
import numpy as np
import os

# Output folder
folder = "../Datasets/Curated Dataset(s)/test_sets"
os.makedirs(folder, exist_ok=True)

# -------------------------------
# Process ONE test dataset
# -------------------------------

def process_test_set(df, name_tag, num_points=15, seed=42):

    # Generate vapor pressure samples
    expanded = generate_vapor_pressures_augmented(
        df,
        num_points=num_points,
        seed=seed
    )
    # Optional: filter very low pressures (same as before)
    expanded = expanded[expanded["log10_P"] >= -4].reset_index(drop=True)

    # -------------------------------
    # Save Chemprop target file
    # -------------------------------
    target_file = os.path.join(folder, f"{name_tag}_VP_targets.csv")

    expanded[["SMILES", "log10_P"]].to_csv(
        target_file,
        index=False
    )

    # -------------------------------
    # Save Chemprop feature files (Temperature)
    # -------------------------------
    feat_csv_file = os.path.join(folder, f"{name_tag}_VP_features.csv")
    feat_npz_file = os.path.join(folder, f"{name_tag}_VP_features.npz")

    expanded[["SMILES", "T_C"]].to_csv(
        feat_csv_file,
        index=False
    )

    # Chemprop requires npz format for features
    T_array = expanded["T_C"].to_numpy().reshape(-1, 1)

    np.savez(feat_npz_file, T_array)

    # -------------------------------
    # Logging
    # -------------------------------
    print(f"\n{name_tag.upper()} TEST SET")
    print(f"Original molecules: {len(df)}")
    print(f"Expanded samples: {len(expanded)}")
    print(f"Saved:")
    print(f"  → {target_file}")
    print(f"  → {feat_npz_file}")

    return expanded

# -------------------------------
# Run for both datasets
# -------------------------------
solids_expanded = process_test_set(
    df_lowVP_solids_cleaned,
    name_tag="lowVP_solids_test"
)

liquids_expanded = process_test_set(
    df_lowVP_liquids_cleaned,
    name_tag="lowVP_liquids_test"
)


In [ ]:
import pandas as pd
df = pd.read_csv('../chemprop_model_output/model34/VP_predictions_complete.csv')
df2 = pd.read_csv('../chemprop_model_output/model34/test_set_predictions/lowVP_liquids_test_predictions_complete.csv')

In [ ]:
# Filter to only the training split
train_df = df[df["split"] == "val"]

# Compute statistics
min_T_C = train_df["T_C"].min()
max_T_C = train_df["T_C"].max()
max_VP_true = train_df["VP_true"].max()
min_VP_true = train_df["VP_true"].min()

# Print results
print(f"Minimum T_C (train): {min_T_C}")
print(f"Maximum T_C (train): {max_T_C}")
print(f"Maximum VP_true (train): {max_VP_true}")
print(f"Mimimum VP_true (train): {min_VP_true}")

In [ ]:
# Count entries with T_C below absolute zero
count_below_abs_zero = (df["T_C"] < -273.15).sum()

print(f"Number of entries with T_C < -273.15: {count_below_abs_zero}")


In [ ]:
# Step 1: Find entries below absolute zero
invalid_rows = df[df["T_C"] < -273.15]

# Step 2: Get SMILES to exclude
invalid_smiles = invalid_rows["SMILES"].unique()

# Step 3: Remove all rows with those SMILES
filtered_df = df[~df["SMILES"].isin(invalid_smiles)].copy()

# Step 4: Find the row with the next lowest temperature
lowest_valid_row = filtered_df.loc[filtered_df["T_C"].idxmin()]

print("Next lowest temperature entry (after removing invalid SMILES):")
print(lowest_valid_row)


In [ ]:
from rdkit import Chem

# Function to canonicalize SMILES
def canonicalize_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            return Chem.MolToSmiles(mol, canonical=True)
    except:
        pass
    return None

# -----------------------------
# Step 1: Restrict df to train + val
# -----------------------------
df_train_val = df[df["split"].isin(["train"])]

# -----------------------------
# Step 2: Get unique SMILES
# -----------------------------
unique_smiles_df1 = df_train_val["SMILES"].dropna().unique()
unique_smiles_df2 = df2["SMILES"].dropna().unique()

# -----------------------------
# Step 3: Canonicalize
# -----------------------------
canon_df1 = {canonicalize_smiles(s) for s in unique_smiles_df1}
canon_df2 = {canonicalize_smiles(s) for s in unique_smiles_df2}

# Remove failed conversions
canon_df1.discard(None)
canon_df2.discard(None)

# -----------------------------
# Step 4: Intersection
# -----------------------------
common_smiles = sorted(canon_df1.intersection(canon_df2))

# Display unique overlapping SMILES
print("Unique overlapping SMILES:")
for smi in common_smiles:
    print(smi)

print(f"\nTotal unique overlapping molecules: {len(common_smiles)}")


In [ ]:
# Add canonical column to df2
df2["SMILES_canon"] = df2["SMILES"].apply(canonicalize_smiles)

# Filter to only overlapping molecules
df2_overlap = df2[df2["SMILES_canon"].isin(common_smiles)].copy()


In [ ]:
df2_overlap.tail(100)